In [9]:
import pandas as pd
import os
import sys
sys.path.append('..')

from datetime import datetime, timedelta
from models.nist.refractive_index import ciddor

def load_interferometer_data(start_date, end_date, data_path='../../data/raw/'):
    """
    Load and process interferometer and environmental data for a given date range.
    
    Args:
        start_date (str): Start date in YYYYMMDD format
        end_date (str): End date in YYYYMMDD format
        data_path (str): Base path to data directory
    
    Returns:
        pd.DataFrame: Processed dataframe with laser counts ratio and calculated refractive index
    
    Raises:
        ValueError: If no complete data files found or data is empty
    """
    # Generate date range
    start = datetime.strptime(start_date, "%Y%m%d")
    end = datetime.strptime(end_date, "%Y%m%d")
    date_range = [start + timedelta(days=i) for i in range((end - start).days + 1)]
    
    # Collect valid file paths
    valid_file_groups = []
    for date in date_range:
        date_str = date.strftime("%Y%m%d")
        
        # Define file paths for all required data types
        file_paths = {
            'counts_ratio': os.path.join(data_path, f'interferometer/counts_ratio_data_{date_str}.csv'),
            'humidity': os.path.join(data_path, f'environmental/humidity_data_{date_str}.csv'),
            'pressure': os.path.join(data_path, f'environmental/pressure_data_{date_str}.csv'),
            'temperature': os.path.join(data_path, f'environmental/temperature_data_{date_str}.csv')
        }
        
        # Check if all required files exist
        if all(os.path.exists(path) for path in file_paths.values()):
            valid_file_groups.append(file_paths)
        else:
            print(f"Skipping {date_str}: Incomplete data files")
    
    if not valid_file_groups:
        raise ValueError("No complete date data found")
    
    # Load and process data
    data_frames = {'counts_ratio': [], 'humidity': [], 'pressure': [], 'temperature': []}
    
    for file_group in valid_file_groups:
        try:
            for data_type, file_path in file_group.items():
                df = pd.read_csv(file_path)
                if 'time' in df.columns:
                    df['time'] = pd.to_datetime(df['time'], utc=True, errors='coerce')
                data_frames[data_type].append(df)
        except Exception as e:
            print(f"Skipping {file_group['counts_ratio']} due to error: {e}")
    
    # Check if any data was loaded
    if not data_frames['counts_ratio']:
        raise ValueError("All data files are invalid or empty")
    
    # Merge data for each type
    merged_data = {}
    for data_type, dfs in data_frames.items():
        df = pd.concat(dfs)
        if data_type == 'counts_ratio':
            # Filter valid counts ratio values (2-3 range)
            df = df[(df['counts_ratio'] >= 2.2584867) & (df['counts_ratio'] <= 3)]
        merged_data[data_type] = df.dropna(subset=['time']).sort_values('time')
    
    # Rename columns to standard names
    column_mapping = {
        'humidity': {'humidity_wm_bme280': 'humidity'},
        'pressure': {'pressure_wm_bme280': 'pressure'},
        'temperature': {'temperature_wm_bme280': 'temperature'}
    }
    
    for data_type, rename_dict in column_mapping.items():
        merged_data[data_type] = merged_data[data_type].rename(columns=rename_dict)
    
    # Merge all environmental data with counts ratio data
    result_df = merged_data['counts_ratio']
    for env_type in ['humidity', 'temperature', 'pressure']:
        result_df = pd.merge_asof(
            result_df,
            merged_data[env_type][['time', env_type]],
            on='time',
            direction='nearest'
        )
    
    # Calculate refractive index at 1762 nm
    def calculate_refractive_index(ratio, temperature, pressure, humidity):
        """Calculate refractive index from counts ratio and environmental parameters"""
        f_1762 = 170.126432e12  # Frequency at 1762 nm
        f_rb = 384.2281145e12   # Rubidium frequency
        n_rb = ciddor(wave=299792458/f_rb*1e9, t=temperature, p=pressure*100, rh=humidity)
        return n_rb * f_rb / f_1762 / ratio
    
    # Apply calculation to all rows using vectorized approach
    result_df['n_1762'] = result_df.apply(
        lambda row: calculate_refractive_index(
            row['counts_ratio'],
            row['temperature'],
            row['pressure'],
            row['humidity']
        ),
        axis=1
    )
    
    return result_df

In [10]:
start_date = '20241230'
end_date = '20250805'

df_training = load_interferometer_data(start_date, end_date)

print(f"loaded {len(df_training)} data in total")
print(df_training.head())

df_training.to_csv(f"../../data/processed/training_data.csv", index=False, encoding="utf-8-sig")

Skipping 20241230: Incomplete data files
Skipping 20241231: Incomplete data files
Skipping 20250112: Incomplete data files
Skipping 20250114: Incomplete data files
Skipping 20250115: Incomplete data files
Skipping 20250117: Incomplete data files
Skipping 20250119: Incomplete data files
Skipping 20250121: Incomplete data files
Skipping 20250122: Incomplete data files
Skipping 20250123: Incomplete data files
Skipping 20250124: Incomplete data files
Skipping 20250127: Incomplete data files
Skipping 20250128: Incomplete data files
Skipping 20250129: Incomplete data files
Skipping 20250130: Incomplete data files
Skipping 20250131: Incomplete data files
Skipping 20250203: Incomplete data files
Skipping 20250205: Incomplete data files
Skipping 20250212: Incomplete data files
Skipping 20250213: Incomplete data files
Skipping 20250214: Incomplete data files
Skipping 20250220: Incomplete data files
Skipping 20250221: Incomplete data files
Skipping 20250222: Incomplete data files
Skipping 2025022

In [11]:
start_date = '20250805'
end_date = '20250821'

df_validation = load_interferometer_data(start_date, end_date)

print(f"loaded {len(df_validation)} data in total")
print(df_validation.head())

df_validation.to_csv(f"../../data/processed/validation_data.csv", index=False, encoding="utf-8-sig")

Skipping 20250821: Incomplete data files
loaded 40897 data in total
                              time  counts_ratio   humidity  temperature  \
0 2025-08-05 00:00:06.757350+00:00      2.258488  31.831336    26.541232   
1 2025-08-05 00:00:18.941835+00:00      2.258488  31.412200    26.541232   
2 2025-08-05 00:00:31.542762+00:00      2.258488  31.831336    26.541232   
3 2025-08-05 00:00:45.281390+00:00      2.258487  31.994175    26.546383   
4 2025-08-05 00:00:56.774360+00:00      2.258488  31.342577    26.536082   

     pressure    n_1762  
0  988.332517  1.000256  
1  988.248014  1.000257  
2  988.288153  1.000257  
3  988.316321  1.000257  
4  988.259986  1.000257  
